In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'data_prep' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'clean'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = RAW_DIR / 'profile.json'
PORTFOLIO_PATH = RAW_DIR / 'portfolio.json'
TRANSCRIPT_PATH = RAW_DIR / 'transcript.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed output directory: {PROCESSED_DIR}')

Project root: D:\SanJose\sem2\252_PROJECT
Processed output directory: D:\SanJose\sem2\252_PROJECT\data\clean


In [2]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as file:
        return [json.loads(line) for line in file if line.strip()]

profile = pd.DataFrame(read_jsonl(PROFILE_PATH))
portfolio = pd.DataFrame(read_jsonl(PORTFOLIO_PATH))
transcript = pd.DataFrame(read_jsonl(TRANSCRIPT_PATH))

In [3]:
transcript_values = pd.json_normalize(transcript['value'].where(transcript['value'].notna(), None))
base_events = transcript.drop(columns='value').reset_index(drop=True)
transcript_values = transcript_values.reset_index(drop=True)

In [4]:
 #The nested payload itself also has two historical spellings. Coalesce them before renaming.
if 'offer id' in transcript_values.columns and 'offer_id' in transcript_values.columns:
    transcript_values['offer_id'] = transcript_values['offer_id'].fillna(transcript_values['offer id'])
    transcript_values = transcript_values.drop(columns='offer id')
elif 'offer id' in transcript_values.columns:
    transcript_values = transcript_values.rename(columns={'offer id': 'offer_id'})
#The raw data uses top-level offer_id for completed events and nested offer id for received/viewed events. 
#coalesce both sources before concatenating to avoid duplicate columns.
if 'offer_id' in base_events.columns and 'offer_id' in transcript_values.columns:
    base_events['offer_id'] = base_events['offer_id'].fillna(transcript_values['offer_id'])
    transcript_values = transcript_values.drop(columns='offer_id')
events = pd.concat([base_events, transcript_values], axis=1)

# Attach offer_type for coupon-level filtering
events = events.merge(portfolio[['id', 'offer_type']], left_on='offer_id', right_on='id', how='left')
events = events.drop(columns='id')

print(f'Loaded profiles: {len(profile):,}')
print(f'Loaded offers: {len(portfolio):,}')
print(f'Loaded events: {len(events):,}')
print('Event columns:', events.columns.tolist())

Loaded profiles: 17,000
Loaded offers: 10
Loaded events: 306,534
Event columns: ['person', 'event', 'time', 'amount', 'offer_id', 'reward', 'offer_type']


Treating Missing Values

In [5]:
# age 118 is a sentinel for customers whose demographics were not supplied so we will treat them as missing.
profile['age'] = profile['age'].replace(118, np.nan)

# Convert the membership date to a real datetime value
profile['became_member_on'] = pd.to_datetime(
    profile['became_member_on'].astype('string'), format='%Y%m%d', errors='coerce'
)


We are filling null age and income values with sample median as dataset is already short

In [6]:
profile['age'] = profile['age'].fillna(profile['age'].median())
profile['income'] = profile['income'].fillna(profile['income'].median())

In [7]:
# Replacing missing gender with an explicit category.
profile['gender'] = profile['gender'].fillna('Unknown')

In [8]:
events['amount'] = events['amount'].fillna(0.0)
events['reward'] = events['reward'].fillna(0.0)
events['offer_id'] = events['offer_id'].fillna('no_offer')
events['offer_type'] = events['offer_type'].fillna('transaction_or_unknown')

In [9]:
profile = profile.drop_duplicates().reset_index(drop=True)
portfolio = portfolio.drop_duplicates(subset='id').reset_index(drop=True)
events = events.drop_duplicates().reset_index(drop=True)

In [10]:
print('Remaining null counts after treatment:')
print('profile:', profile.isna().sum().to_dict())
print('events:', events.isna().sum().to_dict())

Remaining null counts after treatment:
profile: {'gender': 0, 'age': 0, 'id': 0, 'became_member_on': 0, 'income': 0}
events: {'person': 0, 'event': 0, 'time': 0, 'amount': 0, 'offer_id': 0, 'reward': 0, 'offer_type': 0}


In [11]:
transaction_amount = events.loc[events['event'].eq('transaction'), 'amount']
before_outlier_stats = pd.DataFrame({
    'variable': ['profile.age', 'profile.income', 'transaction.amount'],
    'range_min': [profile['age'].min(), profile['income'].min(), transaction_amount.min()],
    'range_max': [profile['age'].max(), profile['income'].max(), transaction_amount.max()],
    'mean': [profile['age'].mean(), profile['income'].mean(), transaction_amount.mean()],
    'median': [profile['age'].median(), profile['income'].median(), transaction_amount.median()]
})

# Round only the displayed copy; the original statistics retain full precision.
display(before_outlier_stats.round(2))
before_outlier_stats.to_csv(PROCESSED_DIR / 'statistics_before_outliers.csv', index=False)

,variable,range_min,range_max,mean,median
0,profile.age,18.00,101.00,54.47,55.00
1,profile.income,30000.00,120000.00,65225.24,64000.00
2,transaction.amount,0.05,1062.28,12.78,8.89


IMP note to self: Should we treat outliers? I AM NOT SURE since data doesnt seem to be that skewed due to outliers, transaction amount maybe

Removing Informational Coupons, since we dont need them for modeling

In [12]:
informational_ids = set(
    portfolio.loc[portfolio['offer_type'].eq('informational'), 'id']
)
portfolio = portfolio.loc[~portfolio['id'].isin(informational_ids)].reset_index(drop=True)
events = events.loc[
    ~events['offer_id'].isin(informational_ids)
].reset_index(drop=True)

print(f'Removed informational offers: {len(informational_ids)}')
print(f'Remaining offers: {len(portfolio):,}')
print(f'Remaining events: {len(events):,}')
print('Remaining offer types:', portfolio['offer_type'].value_counts().to_dict())

Removed informational offers: 2
Remaining offers: 8
Remaining events: 280,071
Remaining offer types: {'bogo': 4, 'discount': 4}
